# 01 - VLM spike (thin wrapper)

Runs `python -m shipdoc spike` for each config on the spike set, one subprocess per config.
All logic is in the package; this notebook only does Drive, secrets, clone, install and
bookkeeping. `PINNED_SHA` in the parameters cell is the code commit it clones; just
Runtime -> Run all.
Re-running Run all skips configs already `done` and resumes partial ones (see notebooks/README.md).

Flow: GPU-hour / compute-unit estimate table (read it before spending quota) -> MANDATORY
5-doc smoke gate per model (img_only config, 6 assertions + a non-blocking cpn==PO warning; a
model that fails is skipped, all failing stops the notebook) -> spike on every config of the
models that passed -> summary -> paired-bootstrap ranking -> keyed-vs-compact format A/B on
rank #1 (one more spike40 run) -> if the top-2 95% CIs overlap, the top-2 run on dev100 and
are re-ranked -> final pick -> DONE.

Format A/B (GG-approved; motivated by Step L root cause #2, rotations among spn/cpn/po): the
six spike configs use the compact output format. The A/B reruns rank #1 in the keyed format
(full field names in declared order, same model and arm) on spike40 and compares the two with
`scripts/format_ab.py`. Choose KEYED if (row F1 keyed > compact AND the paired CI of delta
row F1 excludes 0) OR (rotations keyed < rotations compact AND OVERALL keyed >= compact, where
"OVERALL not worse" = point delta OVERALL >= 0 OR the paired CI of delta OVERALL includes 0);
otherwise COMPACT on speed. dev100 format policy: rank #1 uses the A/B decision; rank #2 has no
A/B of its own, so it uses compact, EXCEPT when the A/B chose keyed: then rank #2 runs keyed too,
because the effect is format-level. If the keyed run fails or OOMs, everything falls back to
compact and the reason is recorded in `ab_status.json`.


In [ ]:
# Parameters. PINNED_SHA = the repo commit this run is pinned to (full 40-char SHA).
PINNED_SHA = "2abf481be6332e24bee0c8e6b0d16fe8cde55ce4"
RUN_GROUP = "spike40"
CONFIGS = [
    "nuextract3_img_ocr_compact",
    "nuextract3_img_only_compact",
    "qwen35_4b_img_ocr_compact",
    "qwen35_4b_img_only_compact",
    "qwen3vl_8b_img_only_compact",
    "qwen3vl_8b_img_ocr_compact"
]
DOCS = "splits/spike40.json"
DEV100_GROUP = "dev100"  # the conditional top-2 run uses run_ids dev100_<config>_<sha7>
DEV100_DOCS = "splits/dev100.json"
SMOKE_DOCS = "splits/smoke5.json"  # 5 docs picked from spike40 by meta tags (shipdoc.smoke)
SMOKE_LIMIT = 5
USE_WANDB = True
WANDB_PROJECT = "shipdoc-extract-debug"  # must be a PRIVATE project (checked below)


## Account

No account check in this public build: Drive is mounted as whichever Google account you sign in
with, and the inputs must be in that account's `MyDrive/shipdoc-extract/` (see the repository
README, "Reproduce from images").


In [ ]:
import subprocess
from pathlib import Path

from google.colab import drive

DRIVE_DIR = Path("/content/drive/MyDrive/shipdoc-extract")
REQUIRED_ZIPS = ["data.zip", "assignment.zip", "ocr_cache.zip"]
RUNS_DIR = DRIVE_DIR / "runs"

drive.mount("/content/drive")

print("Public build: any Google account; inputs are read from", DRIVE_DIR)

missing = [n for n in REQUIRED_ZIPS if not (DRIVE_DIR / n).is_file()]
if missing:
    raise FileNotFoundError(f"Missing in MyDrive/shipdoc-extract/: {', '.join(missing)}")
print("Drive inputs present:", ", ".join(REQUIRED_ZIPS))
RUNS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import os

from google.colab import userdata


def get_secret(name: str, required: bool) -> str | None:
    """Read a Colab secret without ever printing it."""
    try:
        value = userdata.get(name)
    except Exception:  # SecretNotFoundError / NotebookAccessError: both mean "not usable"
        value = None
    if required and not value:
        raise RuntimeError(
            f"Colab secret {name} is missing or notebook access is off "
            "(left sidebar -> key icon -> add it and enable Notebook access)."
        )
    return value or None


WANDB_API_KEY = get_secret("WANDB_API_KEY", required=USE_WANDB)
HF_TOKEN = get_secret("HF_TOKEN", required=False)
print("WANDB_API_KEY:", "set" if WANDB_API_KEY else "not set",
      "| HF_TOKEN:", "set" if HF_TOKEN else "not set (optional)")


In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "FILL_PUBLIC_REPO_URL"
REPO = Path("/content/shipdoc-extract")
if not PINNED_SHA or "FILL" in PINNED_SHA:
    raise ValueError("Set PINNED_SHA in the parameters cell to the commit this run is pinned to.")
if "FILL" in REPO_URL:
    raise ValueError("REPO_URL is a placeholder: the public repository is not published yet.")


def git(*args: str) -> str:
    """Run git in the repo dir (or /content). The repository is public: no credentials at all."""
    cwd = REPO if (REPO / ".git").is_dir() else "/content"
    res = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, check=False)
    if res.returncode:
        raise RuntimeError(f"git {args[0]} failed (exit {res.returncode}): {res.stderr[-800:]}")
    return res.stdout.strip()


if (REPO / ".git").is_dir():
    git("fetch", "origin")
else:
    git("clone", REPO_URL, str(REPO))
git("checkout", PINNED_SHA)
head = git("rev-parse", "HEAD")
pinned_full = git("rev-parse", f"{PINNED_SHA}^{{commit}}")
assert head == pinned_full, f"HEAD {head} != PINNED_SHA {pinned_full}"
print("HEAD =", head)


In [ ]:
import hashlib
import shutil
import zipfile

LOCAL_ZIPS = Path("/content/zips")
LOCAL_ZIPS.mkdir(exist_ok=True)
# data.zip holds data/, assignment.zip holds assignment/, ocr_cache.zip holds ocr_cache/.
TARGETS = {"data.zip": REPO, "assignment.zip": REPO, "ocr_cache.zip": Path("/content")}
for name, dest in TARGETS.items():
    local = LOCAL_ZIPS / name
    if not local.is_file() or local.stat().st_size != (DRIVE_DIR / name).stat().st_size:
        shutil.copyfile(DRIVE_DIR / name, local)  # local disk: Drive reads are slow
    with zipfile.ZipFile(local) as zf:
        top = zf.namelist()[0].split("/")[0]
        if not (dest / top).is_dir():
            zf.extractall(dest)
    print(f"{name}: unzipped to {dest / top}")

# Verify the OCR cache against its own SHA256SUMS (paths are relative to ocr_cache/).
OCR_CACHE = Path("/content/ocr_cache")
bad = []
sums = (OCR_CACHE / "SHA256SUMS").read_text(encoding="utf-8").splitlines()
for line in sums:
    digest, _, rel = line.partition("  ")
    f = OCR_CACHE / rel
    if not f.is_file() or hashlib.sha256(f.read_bytes()).hexdigest() != digest:
        bad.append(rel)
if bad:
    raise RuntimeError(f"ocr_cache SHA256SUMS: {len(bad)} bad/missing files, e.g. {bad[:5]}")
print(f"ocr_cache OK: {len(sums)} files verified")


In [ ]:
import collections
import json
import subprocess

PY = str(REPO / ".venv" / "bin" / "python")
ENV = {
    **os.environ,
    "SHIPDOC_PROFILE": "colab",
    "HF_HOME": "/content/hf_cache",
    "SHIPDOC_RUNS_DIR": str(RUNS_DIR),
    "SHIPDOC_DATA_DIR": str(REPO / "data"),
    "SHIPDOC_ASSIGNMENT_DIR": str(REPO / "assignment"),
    "SHIPDOC_OCR_CACHE": "/content/ocr_cache",
    "SHIPDOC_TMP_DIR": "/content/tmp",
    "UV_CACHE_DIR": "/content/uv_cache",
    "WANDB_PROJECT": WANDB_PROJECT,
    "WANDB_RUN_GROUP": RUN_GROUP,
}
if WANDB_API_KEY:
    ENV["WANDB_API_KEY"] = WANDB_API_KEY
if HF_TOKEN:
    ENV["HF_TOKEN"] = HF_TOKEN
Path(ENV["SHIPDOC_TMP_DIR"]).mkdir(exist_ok=True)


def run_stream(
    cmd: list[str], tail: int = 40, env: dict | None = None
) -> tuple[int, list[str]]:
    """Run a command, echo its output live, return (exit code, last `tail` lines)."""
    last: collections.deque[str] = collections.deque(maxlen=tail)
    with subprocess.Popen(
        cmd, cwd=REPO, env=env or ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    ) as proc:
        for line in proc.stdout:
            print(line, end="")
            last.append(line.rstrip("\n"))
    return proc.returncode, list(last)


if subprocess.run(["uv", "--version"], capture_output=True, check=False).returncode:
    subprocess.run(["pip", "install", "-q", "uv"], check=True)
rc, _ = run_stream(["uv", "sync", "--frozen", "--group", "vlm", "--python", "3.11"])
assert rc == 0, f"uv sync failed (exit {rc})"

subprocess.run(["nvidia-smi"], check=False)
PROBE = (
    "import json, importlib.metadata as m, torch;"
    "print(json.dumps({'torch': torch.__version__, 'cuda': torch.cuda.is_available(),"
    " 'arch': torch.cuda.get_arch_list(),"
    " **{p: m.version(p) for p in ('transformers', 'bitsandbytes', 'xgrammar')}}))"
)
info = json.loads(subprocess.run([PY, "-c", PROBE], env=ENV, capture_output=True, text=True,
                                 check=True).stdout.strip().splitlines()[-1])
print(info)
assert info["cuda"], "No CUDA device: Runtime -> Change runtime type -> T4 GPU."
# T4 is sm_75: need a native sm_75 kernel or PTX (compute_XX with XX <= 75) it can JIT.
ok = "sm_75" in info["arch"] or any(
    a.startswith("compute_") and int(a.split("_")[1]) <= 75 for a in info["arch"]
)
assert ok, f"torch build has no sm_75 support: {info['arch']}"


In [ ]:
# Log in to W&B and check the debug project is PRIVATE.
WANDB_CHECK = """
import os, sys
import wandb

wandb.login(key=os.environ["WANDB_API_KEY"], verify=True)
api = wandb.Api()
entity, name = api.default_entity, os.environ["WANDB_PROJECT"]
print("W&B entity:", entity, "| project:", name)
try:
    attrs = dict(getattr(api.project(name, entity), "attrs", {}) or {})
except Exception as exc:  # project absent, or this wandb version lacks api.project
    attrs = {}
    print("project lookup:", type(exc).__name__)
KEYS = ("access", "public", "privacy")
flags = {k: v for k, v in attrs.items() if any(s in k.lower() for s in KEYS)}
print("access-related fields:", flags or "none exposed by the public API")
public = any(v is True and "public" in k.lower() for k, v in flags.items()) or any(
    isinstance(v, str) and v.upper() == "PUBLIC" for v in flags.values()
)
private = any(v is False and "public" in k.lower() for k, v in flags.items()) or any(
    isinstance(v, str) and v.upper() in ("PRIVATE", "TEAM") for v in flags.values()
)
if public:
    sys.exit("PROJECT IS PUBLIC: set it to Private in W&B before running.")
if not private:
    print(f"ACTION FOR GG: the W&B public API does not expose project access. Open "
          f"https://wandb.ai/{entity}/{name}/settings and verify Visibility = Private "
          f"(create the project as Private first, or set the account default project privacy "
          f"to Private; an auto-created project takes the default).")
else:
    print("project access: private OK")
"""
if USE_WANDB:
    res = subprocess.run([PY, "-c", WANDB_CHECK], env=ENV, check=False)
    assert res.returncode == 0, "W&B login or privacy check failed (see output above)."


In [ ]:
# GPU-hour and compute-unit (CU) estimate BEFORE any model runs: read it to budget.
# No confirmation prompt (Run all must not block): interrupt the run yourself if it is too high.
import importlib.util
import subprocess

_spec = importlib.util.spec_from_file_location("gpu_estimate", REPO / "scripts" / "gpu_estimate.py")
ge = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(ge)

SPEED = json.loads((REPO / "configs" / "spike_speed.json").read_text(encoding="utf-8"))
LABELS = REPO / "data" / "dev" / "labels"
spike_ids = json.loads((REPO / DOCS).read_text(encoding="utf-8"))
dev100_ids = json.loads((REPO / DEV100_DOCS).read_text(encoding="utf-8"))
smoke_ids = json.loads((REPO / SMOKE_DOCS).read_text(encoding="utf-8"))


def read_gpu_name():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=30, check=False,
        ).stdout.strip().splitlines()
    except (OSError, subprocess.TimeoutExpired):
        return None
    return out[0].strip() if out else None


GPU_NAME = read_gpu_name()
print("Runtime GPU (nvidia-smi):", GPU_NAME)
print(ge.report(CONFIGS, SPEED, ge.count_pages(spike_ids, LABELS),
                ge.count_pages(dev100_ids, LABELS), len(spike_ids),
                ge.count_pages(smoke_ids, LABELS), GPU_NAME))
if ge.order_by_estimate(CONFIGS, SPEED) != CONFIGS:
    print("NOTE: CONFIGS order differs from the estimated cheapest-first order.")


In [ ]:
# MANDATORY smoke gate (the fixes under test are unverified on GPU): per model, run its
# img_only compact config on the 5 smoke docs and assert 6 properties of the output
# (shipdoc.smoke). A model that fails is skipped (skipped_smoke_fail) and the rest continue;
# if every model fails the notebook stops here. Resumable via smoke_status.json on Drive.
import sys
import time

import yaml

_smoke_path = REPO / "src" / "shipdoc" / "smoke.py"
_spec = importlib.util.spec_from_file_location("shipdoc_smoke", _smoke_path)
smoke = importlib.util.module_from_spec(_spec)
sys.modules["shipdoc_smoke"] = smoke  # dataclasses resolves annotations through sys.modules
_spec.loader.exec_module(smoke)

SHA7 = PINNED_SHA[:7]
SMOKE_RUNS = RUNS_DIR / "smoke"
SMOKE_STATUS_PATH = RUNS_DIR / "smoke_status.json"
smoke_status = json.loads(SMOKE_STATUS_PATH.read_text()) if SMOKE_STATUS_PATH.is_file() else {}
SMOKE_ENV = {**ENV, "SHIPDOC_RUNS_DIR": str(SMOKE_RUNS)}
SMOKE_ENV.pop("WANDB_API_KEY", None)  # smoke runs never log to W&B
MODELS = list(dict.fromkeys(ge.model_key(c) for c in CONFIGS))
print(f"SMOKE GATE: {len(smoke_ids)} docs {smoke_ids}, models {MODELS}")


def save_smoke_status() -> None:
    tmp = SMOKE_STATUS_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(smoke_status, indent=1))
    tmp.replace(SMOKE_STATUS_PATH)  # atomic: a disconnect never leaves a half-written file


for model in MODELS:
    config = f"{model}_img_only_compact"
    run_id = f"smoke_{config}_{SHA7}"
    prev = smoke_status.get(model, {})
    if prev.get("state") == "passed" and prev.get("run_id") == run_id:
        print(f"SKIP smoke {model}: already passed ({run_id})")
        continue
    cfg_path = f"configs/spike_{config}.yaml"
    run_dir = SMOKE_RUNS / run_id
    t0 = time.time()
    rc, tail, checks, warns, error = 0, [], [], [], None
    try:
        max_new = int(yaml.safe_load((REPO / cfg_path).read_text())["max_new_tokens"])
        if not (run_dir / "metrics.json").is_file():  # a finished run is re-checked, not re-run
            print(f"\n=== {run_id} ===", flush=True)
            cmd = [PY, "-m", "shipdoc", "spike", "--config", cfg_path, "--docs", SMOKE_DOCS,
                   "--split", "dev", "--run-id", run_id, "--resume", "--limit", str(SMOKE_LIMIT)]
            rc, tail = run_stream(cmd, env=SMOKE_ENV)
        if rc == 0:
            checks = smoke.load_and_check(run_dir, LABELS, max_new)
            warns = smoke.load_warnings(run_dir)  # non-blocking: never raises, never fails
    except (OSError, KeyError, ValueError) as exc:  # fail closed: could not verify = fail
        rc, error = 1, f"{type(exc).__name__}: {exc}"
    if rc != 0:
        state = "error"
        print(f"SMOKE ERROR {model}: exit {rc} {error or ''}; last output: {tail[-3:]}")
    else:
        state = "passed" if all(c.passed for c in checks) else "failed"
        print(smoke.format_table(model, checks, warns))
    smoke_status[model] = {
        "state": state,
        "config": config,
        "run_id": run_id,
        "exit_code": rc,
        "seconds": round(time.time() - t0, 1),
        "checks": [{"name": c.name, "passed": c.passed, "detail": c.detail} for c in checks],
        "warnings": [{"name": w.name, "count": w.count, "detail": w.detail} for w in warns],
        "error": error,
        "tail": tail[-5:] if rc else [],
    }
    save_smoke_status()

SMOKE_FAILED_MODELS = {m for m in MODELS if smoke_status[m]["state"] != "passed"}
print("\nSMOKE SUMMARY")
for m in MODELS:
    v = smoke_status[m]
    bad = [c["name"] for c in v["checks"] if not c["passed"]]
    cpn_po = next((w["count"] for w in v.get("warnings", []) if w["name"] == "cpn_equals_po"), None)
    print(f"  {m.ljust(14)} {v['state'].upper().ljust(7)} {', '.join(bad) or v['error'] or ''}"
          f"  [warn cpn==PO rows: {cpn_po}]")
print("smoke status file:", SMOKE_STATUS_PATH)
if len(SMOKE_FAILED_MODELS) == len(MODELS):
    raise RuntimeError(
        "SMOKE GATE FAILED FOR ALL MODELS: stopping before the full run. GG: report back with "
        f"{SMOKE_STATUS_PATH} and the per-assertion tables printed above (see notebooks/README.md)."
    )
if SMOKE_FAILED_MODELS:
    print(f"WARNING: skipping all configs of {sorted(SMOKE_FAILED_MODELS)} (skipped_smoke_fail).")


In [ ]:
import json
import re
import time

SHA7 = PINNED_SHA[:7]
STATUS_PATH = RUNS_DIR / f"{RUN_GROUP}_status.json"
OOM_RE = re.compile(r"out of memory|OutOfMemoryError", re.IGNORECASE)


def run_id_for(config: str, group: str = RUN_GROUP) -> str:
    return f"{group}_{config}_{SHA7}"


def run_group(group: str, configs: list[str], docs: str, status_name: str | None = None) -> dict:
    """Run `configs` on `docs` as <group>_<config>_<sha7>; skip done runs, resume partial.

    Configs of a model that failed the smoke gate are recorded as skipped_smoke_fail, not run.
    The status file is <status_name or group>_status.json.
    """
    status_path = RUNS_DIR / f"{status_name or group}_status.json"
    status = json.loads(status_path.read_text()) if status_path.is_file() else {}
    ENV["WANDB_RUN_GROUP"] = group
    for config in configs:
        run_id = run_id_for(config, group)
        if config.rsplit("_img_", 1)[0] in SMOKE_FAILED_MODELS:
            status[run_id] = {"state": "skipped_smoke_fail", "exit_code": None, "reason": "smoke"}
            print(f"SKIP {run_id}: model failed the smoke gate (skipped_smoke_fail)")
            tmp = status_path.with_suffix(".tmp")
            tmp.write_text(json.dumps(status, indent=1))
            tmp.replace(status_path)
            continue
        metrics = RUNS_DIR / run_id / "metrics.json"
        if status.get(run_id, {}).get("state") == "done" and metrics.is_file():
            print(f"SKIP {run_id}: already done")
            continue
        cfg_path = f"configs/spike_{config}.yaml"
        cmd = [PY, "-m", "shipdoc", "spike", "--config", cfg_path, "--docs", docs, "--split",
               "dev", "--run-id", run_id, "--resume"] + (["--wandb"] if USE_WANDB else [])
        print(f"\n=== {run_id} ===", flush=True)
        t0 = time.time()
        if not (REPO / cfg_path).is_file():
            rc, tail = 127, [f"config not found: {cfg_path}"]
            print(tail[0])
        else:
            # One subprocess per config: its CUDA memory is fully released when it exits.
            rc, tail = run_stream(cmd)
        state = "done" if rc == 0 else "failed"
        status[run_id] = {
            "state": state,
            "exit_code": rc,
            "reason": "oom" if rc and OOM_RE.search("\n".join(tail)) else None,
            "seconds": round(time.time() - t0, 1),
            "tail": tail[-5:] if rc else [],
        }
        tmp = status_path.with_suffix(".tmp")
        tmp.write_text(json.dumps(status, indent=1))
        tmp.replace(status_path)  # atomic: a disconnect never leaves a half-written file
        print(f"--- {run_id}: {state} (exit {rc}) ---")
    print("\nstatus file:", status_path)
    return status


status = run_group(RUN_GROUP, CONFIGS, DOCS)


In [ ]:
def ci_str(v: dict) -> str:
    return f"{v['point']:.3f} [{v['lo']:.3f}, {v['hi']:.3f}]"


def num(v, spec: str = ".3f") -> str:
    # null is a real value in metrics.json (mock runs have no GPU): show it, never hide it.
    return "null" if v is None else format(v, spec)


rows = []
for config in CONFIGS:
    run_id = run_id_for(config)
    path = RUNS_DIR / run_id / "metrics.json"
    if not path.is_file():
        state = status.get(run_id, {}).get("state", "not run")
        rows.append([config, f"NO METRICS (status: {state})"] + [""] * 8)
        continue
    m = json.loads(path.read_text())
    vram = m["peak_vram_max_bytes"]
    rows.append([
        config,
        str(m["n_docs"]),
        ci_str({"point": m["OVERALL"], **{k: m["OVERALL_ci95"][k] for k in ("lo", "hi")}}),
        ci_str(m["scanned"]["ci95"]["OVERALL"]),
        num(m["digital"]["OVERALL"]),
        num(m["false_fill_rate"]),
        num(m["json_validity_rate"]),
        f"{m['s_per_page_mean']:.2f} ({m['s_per_page_p95']:.2f})",
        "null" if vram is None else f"{vram / 1e9:.2f}",
        num(m["n_visual_tokens_mean"], ".1f"),
    ])

header = ["config", "n_docs", "OVERALL [CI]", "scanned OVERALL [CI]", "digital OVERALL",
          "false_fill", "json_validity", "s/page mean (p95)", "peak VRAM GB", "visual tokens mean"]
widths = [max(len(str(r[i])) for r in [header] + rows) for i in range(len(header))]
for r in [header] + rows:
    print("  ".join(str(c).ljust(w) for c, w in zip(r, widths, strict=True)))
failed = [k for k, v in status.items() if v["state"] != "done" and k.startswith(RUN_GROUP + "_")]
if failed:
    print("\nFAILED configs:", failed, "(details in", STATUS_PATH, ")")


In [ ]:
# Rank on OVERALL; paired bootstrap (2000, seed 42) of the top two on shared docs.
# Overlapping individual 95% CIs -> run both on dev100 (spec Phase 2.3): scripts/paired_rank.py.


def rank_group(group: str, configs: list[str], docs: str) -> dict:
    out = RUNS_DIR / f"{group}_rank.json"
    out.unlink(missing_ok=True)  # never read a stale ranking from an earlier session
    cmd = [PY, "-m", "scripts.paired_rank", "--runs-dir", str(RUNS_DIR), "--group", group,
           "--sha7", SHA7, "--docs", docs, "--split", "dev", "--out", str(out),
           "--configs", *configs]
    rc, _ = run_stream(cmd)
    if rc not in (0, 2) or not out.is_file():  # 2 = fewer than 2 usable runs (fail closed)
        raise RuntimeError(f"paired_rank failed (exit {rc}); see output above")
    return json.loads(out.read_text())


def show_rank(r: dict, label: str) -> None:
    print(f"\n{label}: decision = {r['decision']}")
    for e in r.get("excluded", []):
        print(f"  excluded {e['config']}: {e['reason']}")
    if r["decision"] == "insufficient_runs":
        print("  fewer than 2 complete runs: no ranking, no pick")
        return
    for i, x in enumerate(r["ranking"], 1):
        lo, hi = x["ci95"]
        print(f"  {i}. {x['config']}  OVERALL {x['OVERALL']:.3f} [{lo:.3f}, {hi:.3f}]"
              f"  false_fill {x['false_fill_rate']:.3f}  (n={x['n_docs']})")
    lo, hi = r["ci95"]
    print(f"  top-2 {r['top2']}: paired delta (top1 - top2) = {r['delta']:.3f} "
          f"[{lo:.3f}, {hi:.3f}], P(delta <= 0) = {r['p_le_0']:.3f}")
    print(f"  individual 95% CIs overlap: {r['cis_overlap']}; "
          f"paired-delta CI excludes 0: {r['delta_ci_excludes_0']}")


rank = rank_group(RUN_GROUP, CONFIGS, DOCS)
show_rank(rank, f"{RUN_GROUP} ranking")


In [ ]:
# Keyed vs compact format A/B on rank #1 (GG-approved; Step L root cause #2: rotations among
# spn/cpn/po). The keyed config is rank #1's name without `_compact` (same model and arm, full
# field names in declared order). Same run_stream / resume / status mechanics as the spike loop,
# own status file ab_status.json, run_id ab_keyed_<keyed config>_<sha7>. Decision rule and the
# "OVERALL not worse" definition: scripts/format_ab.py (copied in the title cell and spec.md).
# A failed or OOM keyed run, or a failed comparison, falls back to compact and records why.
AB_GROUP = "ab_keyed"
AB_STATUS_NAME = "ab"
AB_RESULT_PATH = RUNS_DIR / "ab_result.json"


def keyed_config(config: str) -> str:
    """The keyed (non-compact) config of the same model and arm."""
    return config.removesuffix("_compact")


def apply_format(config: str, fmt: str) -> str:
    """`config` in the winning format; compact if keyed is chosen but has no config file."""
    if fmt != "keyed":
        return config
    keyed = keyed_config(config)
    if not (REPO / "configs" / f"spike_{keyed}.yaml").is_file():
        print(f"NOTE: no keyed config for {config}: it stays compact")
        return config
    return keyed


ab_decision, ab_reason, ab_result = "compact", "no ranking: A/B not run", None
if rank["decision"] != "insufficient_runs":
    top = rank["ranking"][0]["config"]
    keyed = keyed_config(top)
    model = keyed.rsplit("_img_", 1)[0]
    arm = "img_ocr" if keyed.endswith("_img_ocr") else "img_only"
    if keyed == top or not (REPO / "configs" / f"spike_{keyed}.yaml").is_file():
        ab_reason = f"no keyed counterpart of {top}: A/B not run"
    else:
        print(f"A/B: rank #1 = {top}; running keyed {keyed} on {DOCS}")
        ab_status = run_group(AB_GROUP, [keyed], DOCS, status_name=AB_STATUS_NAME)
        ab_run_id = run_id_for(keyed, AB_GROUP)
        st = ab_status.get(ab_run_id, {})
        if st.get("state") != "done":
            ab_reason = (f"keyed run {st.get('state', 'not run')} (exit {st.get('exit_code')}, "
                         f"reason {st.get('reason')}): compact fallback")
        else:
            AB_RESULT_PATH.unlink(missing_ok=True)  # never read a stale result
            cmd = [PY, "-m", "scripts.format_ab", "--compact-dir", str(RUNS_DIR / run_id_for(top)),
                   "--keyed-dir", str(RUNS_DIR / ab_run_id), "--model", model, "--arm", arm,
                   "--split", "dev", "--out", str(AB_RESULT_PATH), "--table"]
            rc, _ = run_stream(cmd)
            if rc != 0 or not AB_RESULT_PATH.is_file():
                ab_reason = f"format_ab failed (exit {rc}): compact fallback"
            else:
                ab_result = json.loads(AB_RESULT_PATH.read_text())
                ab_decision, ab_reason = ab_result["decision"], ab_result["reason"]
print(f"A/B DECISION: {ab_decision} ({ab_reason})")


In [ ]:
# Conditional: only when the spike top-2 CIs overlap. Same resume/status mechanics
# (run_ids dev100_<config>_<sha7>, status file dev100_status.json), then re-rank on dev100.
# Format policy: rank #1 uses the A/B decision; rank #2 has no A/B, so it is compact unless the
# A/B chose keyed, in which case it runs keyed too (the effect is format-level). Both therefore
# use ab_decision.
final_pick = None
if rank["decision"] == "run_dev100":
    top2 = [apply_format(c, ab_decision) for c in rank["top2"]]
    print(f"Top-2 CIs overlap: running {top2} ({ab_decision} format) on {DEV100_DOCS}")
    dev_status = run_group(DEV100_GROUP, top2, DEV100_DOCS)
    rank100 = rank_group(DEV100_GROUP, top2, DEV100_DOCS)
    show_rank(rank100, f"{DEV100_GROUP} ranking")
    if rank100["decision"] != "insufficient_runs":
        final_pick = rank100["ranking"][0]["config"]
elif rank["decision"] == "pick_top1":
    final_pick = apply_format(rank["ranking"][0]["config"], ab_decision)
    print("Top-2 CIs do not overlap: pick top-1, dev100 skipped.")
else:
    print("No pick: fewer than 2 complete spike runs (see excluded runs above).")
print(f"FINAL PICK: {final_pick}")


In [ ]:
print(f"\nDONE {RUN_GROUP}_{SHA7} decision={rank['decision']} final_pick={final_pick}"
      f" ab_decision={ab_decision}")
